# Section 05: 文本摘要（Summarization）核心总结

## 任务定义
**文本摘要** = 给定长文本，生成简短的摘要。同样是 Seq2Seq 任务，但语义压缩更强。

分类：
- **Extractive（抽取式）**：从原文挑选句子组合（如 baseline 的 3句摘要）
- **Abstractive（生成式）**：模型生成新词句，本节重点

## 本节任务
用 `google/mt5-small`（多语言 T5 小模型）微调：
- 输入：Amazon 英文/西班牙文书评正文（`review_body`）
- 输出：评论标题（`review_title`）—— 这是自然语言摘要的代理任务

## 与翻译（section-04）的异同
| 方面 | 翻译 | 摘要 |
|------|------|------|
| 架构 | Seq2Seq | Seq2Seq（相同）|
| 评估指标 | BLEU | ROUGE |
| 输入长度 | 短句 | 长文本（需截断）|
| 多语言 | 两种语言 | 单语言 or 多语言混合 |
| 基准对比 | 现有翻译模型 | 3句抽取式摘要 |

---
## 第一步：数据准备 — 多语言混合数据集

In [ ]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

# 加载英文和西班牙文亚马逊书评
spanish_dataset = load_dataset("amazon_reviews_multi", "es")
english_dataset = load_dataset("amazon_reviews_multi", "en")

# 只保留书评（book + digital_ebook_purchase），过滤其他品类
def filter_books(example):
    return example["product_category"] in ["book", "digital_ebook_purchase"]

english_books = english_dataset.filter(filter_books)
spanish_books = spanish_dataset.filter(filter_books)

# 合并英文+西班牙文，并打乱
books_dataset = DatasetDict()
for split in english_books.keys():
    books_dataset[split] = concatenate_datasets(
        [english_books[split], spanish_books[split]]
    ).shuffle(seed=42)

# 过滤标题太短的样本（标题 ≤ 2词的不适合做摘要目标）
books_dataset = books_dataset.filter(lambda x: len(x["review_title"].split()) > 2)

# 查看样本结构
print("特征:", list(english_dataset["train"].features.keys()))
# ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', ...]

---
## 第二步：mT5 的分词特点

mT5 使用 SentencePiece tokenizer，与 BERT 的 WordPiece 不同：
- 不需要 `as_target_tokenizer()`（mT5 的 tokenizer 对源和目标语言统一处理）
- 用 `▁` 表示词的起始（空格前缀），例如 `▁Hung er ▁Games`

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 演示 mT5 的 SentencePiece 分词
inputs = tokenizer("I loved reading the Hunger Games!")
print(tokenizer.convert_ids_to_tokens(inputs.input_ids))
# ['▁I', '▁', 'loved', '▁reading', '▁the', '▁Hung', 'er', '▁Games', '</s>']
# 注意：'▁' 表示词前的空格，</s> 是 EOS

In [ ]:
max_input_length = 512   # 书评可能很长
max_target_length = 30   # 标题通常很短

def preprocess_function(examples):
    """
    与翻译的预处理几乎相同，区别：
    - 输入是 review_body（长文本），目标是 review_title（短标题）
    - mT5 无需 as_target_tokenizer()
    """
    model_inputs = tokenizer(
        examples["review_body"],
        max_length=max_input_length,
        truncation=True,
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["review_title"],
            max_length=max_target_length,
            truncation=True,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = books_dataset.map(preprocess_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(books_dataset["train"].column_names)

---
## 第三步：评估指标 — ROUGE

**ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**：摘要任务标准指标

| 指标 | 说明 |
|------|------|
| ROUGE-1 | 1-gram（单词）重叠 |
| ROUGE-2 | 2-gram（词对）重叠 |
| ROUGE-L | 最长公共子序列（LCS），考虑语序 |
| ROUGE-Lsum | 按句子计算 LCS，再取平均 |

**ROUGE vs BLEU**：
- BLEU 侧重 Precision（预测词有多少在参考中），适合翻译
- ROUGE 侧重 Recall（参考词有多少被预测到），适合摘要

In [ ]:
import evaluate
import nltk
import numpy as np

rouge_score = evaluate.load("rouge")
nltk.download("punkt")

# 演示 ROUGE 计算
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"
scores = rouge_score.compute(predictions=[generated_summary], references=[reference_summary])
print(f"ROUGE-1: {scores['rouge1']:.2f}")
print(f"ROUGE-2: {scores['rouge2']:.2f}")
print(f"ROUGE-L: {scores['rougeL']:.2f}")

In [ ]:
# 建立 Baseline：用文章前3句话作为摘要（抽取式）
from nltk.tokenize import sent_tokenize
import pandas as pd

def three_sentence_summary(text):
    return "\n".join(sent_tokenize(text)[:3])

def evaluate_baseline(dataset, metric):
    summaries = [three_sentence_summary(text) for text in dataset["review_body"]]
    return metric.compute(predictions=summaries, references=dataset["review_title"])

score = evaluate_baseline(books_dataset["validation"], rouge_score)
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = {rn: round(score[rn].mid.fmeasure * 100, 2) for rn in rouge_names}
print("3句抽取式基准:", rouge_dict)
# {'rouge1': 16.74, 'rouge2': 8.83, 'rougeL': 15.60, 'rougeLsum': 15.96}

In [ ]:
def compute_metrics(eval_pred):
    """
    ROUGE 计算时，需要在每句话后加换行符（rougeLsum 要求）
    """
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE 期望每个句子后面有换行
    decoded_preds  = ["\n".join(sent_tokenize(pred.strip()))  for pred in decoded_preds]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]

    result = rouge_score.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )
    # 提取 F1 中位数，乘以 100
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    return {k: round(v, 4) for k, v in result.items()}

---
## 第四步：训练

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

batch_size = 8
num_train_epochs = 8
logging_steps = len(tokenized_datasets["train"]) // batch_size

args = Seq2SeqTrainingArguments(
    output_dir=f"mt5-small-finetuned-amazon-en-es",
    evaluation_strategy="epoch",
    learning_rate=5.6e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,  # 必须！摘要用 generate() 评估
    logging_steps=logging_steps,
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# trainer.train()
# 训练后 ROUGE-1 约 16.97，超过抽取式基准（16.74）

---
## 第五步：推理验证

In [ ]:
from transformers import pipeline

hub_model_id = "huggingface-course/mt5-small-finetuned-amazon-en-es"
summarizer = pipeline("summarization", model=hub_model_id)

# 英文评论
review_en = "Nothing special at all about this product... the book is too small and stiff and hard to write in."
title_en = "Not impressed at all... buy something else"
summary_en = summarizer(review_en)[0]["summary_text"]
print(f"原始标题: {title_en}")
print(f"生成摘要: {summary_en}")

# 西班牙文评论（模型支持多语言）
review_es = "Es una trilogia que se hace muy facil de leer. Me ha gustado, no me esperaba el final para nada"
title_es = "Buena literatura para adolescentes"
summary_es = summarizer(review_es)[0]["summary_text"]
print(f"\n原始标题(ES): {title_es}")
print(f"生成摘要(ES): {summary_es}")

---
## 总结

### 与翻译的代码差异（几乎相同的流程）

```python
# 翻译（section-04）          摘要（section-05）
源: en_sentence              源: review_body（长文本）
目标: fr_sentence            目标: review_title（短标题）
指标: BLEU                   指标: ROUGE
模型: MarianMT               模型: mT5（多语言T5）
max_input=128                max_input=512
max_target=128               max_target=30
```

### ROUGE 各子指标含义
```
ROUGE-1:   单词重叠 → 基本词汇覆盖
ROUGE-2:   词对重叠 → 短语质量
ROUGE-L:   最长公共子序列 → 整体流畅度
ROUGE-Lsum:按句 LCS → 多句摘要质量
```

### 摘要任务的特殊处理
```python
# compute_metrics 中，每句话要加 \n（rougeLsum 需要）
decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in decoded_preds]
```

### 典型 ROUGE 分数参考
| 方法 | ROUGE-1 | ROUGE-2 |
|------|---------|----------|
| 3句抽取式（基准）| 16.74 | 8.83 |
| mT5 微调后 | ~16.97 | ~8.30 |